# E2E Part 2 — WhisperRT and overlap baseline

This notebook implements Sections 3–5 of `docx/steps.md`. Run it only after `english-streaming-asr.ipynb` and after saving the Phase 3/5 artifact bundle. It requires Internet, Hugging Face access, and a verified upstream WhisperRT runtime.

In [ ]:
%cd /kaggle/working/English-Streaming-ASR
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / 'src').is_dir()
print(ROOT)

## 3.1 Hugging Face access

First request/confirm access to `MLSpeech/WhisperRT-Streaming` on Hugging Face. Then add the token as a Kaggle secret named `HF_TOKEN`. This cell uses the secret without printing it.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
print('Hugging Face login completed.')

In [ ]:
from huggingface_hub import hf_hub_download

whisperrt_checkpoint = hf_hub_download(
    repo_id='MLSpeech/WhisperRT-Streaming',
    filename='small_300.pt',
)
print(whisperrt_checkpoint)

## 3.2 Required runtime compatibility gate

The project adapter expects the `whisper_rt` module, `whisper_rt.audio.SpectrogramStream`, and `whisper_rt.StreamingWhisper`. All checks must pass before running Phase 1. If they fail, stop: install/pin the official runtime that supplies this API or make a documented correction-branch adapter change. Do not substitute offline Whisper.

In [ ]:
!python -c "import whisper_rt; print(whisper_rt.__file__)"
!python -c "from whisper_rt.audio import SpectrogramStream; print(SpectrogramStream)"
!python -c "from whisper_rt import StreamingWhisper; print(StreamingWhisper)"
!python -c "import torch; print(torch.__version__); print(torch.version.cuda)"

## 3.3 Record the verified model runtime

Run this after the compatibility gate passes. Keep the output with the experiment artifacts. The exact upstream installation command is intentionally not guessed here; use the official release instructions for the version that passed the previous cell.

In [ ]:
!python -m pip freeze | grep -Ei 'torch|whisper|careless|huggingface' | sort
!python -c "import whisper_rt; print(whisper_rt.__file__)"

## 3.4 One-example clean WhisperRT baseline

Only run this after the compatibility gate passes. Expected outputs are timestamped JSONL predictions, metric JSON, report Markdown, and run metadata under `/kaggle/working/outputs/`.

In [ ]:
!python scripts/run_streaming.py \
  --config configs/whisperrt_baseline.yaml \
  --max-samples 1

!ls -lt /kaggle/working/outputs/predictions | head
!ls -lt /kaggle/working/outputs/metrics | head
!ls -lt /kaggle/working/outputs/reports | head

## 3.5 Small clean baseline

Read the one-example report first. If it contains a non-empty prediction and measured timing, run a small development baseline. Preserve the resulting prediction JSONL, metrics, report, config, commit, checkpoint path, and installed package versions.

In [ ]:
!python scripts/run_streaming.py \
  --config configs/whisperrt_baseline.yaml \
  --max-samples 20

# 4. Real synthetic-overlap development dataset

The 12-item demo set is an integration fixture. This section creates a small, separately named development dataset. Do not use final-test language for it.

In [ ]:
from pathlib import Path

Path('kaggle_overlap_dev.yaml').write_text('''\
generation:
  num_samples: 20
  max_source_samples: 100
output:
  root: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev
''')
print(Path('kaggle_overlap_dev.yaml').read_text())

In [ ]:
!python scripts/generate_overlap.py \
  --config configs/overlap_dataset.yaml \
  --override kaggle_overlap_dev.yaml

!find outputs/synthetic_overlap_dev -maxdepth 3 -type f | sort | sed -n '1,80p'

Publish `outputs/synthetic_overlap_dev/` as a private Kaggle Dataset before expensive follow-up runs. In later sessions, attach it and replace the working path below with `/kaggle/input/<your-dataset-slug>`.

# 5. Phase 4 — real WhisperRT overlap baseline

In [ ]:
from pathlib import Path

Path('kaggle_phase4.yaml').write_text('''\
data:
  manifest_root: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev
  conditions: [test]
''')
print(Path('kaggle_phase4.yaml').read_text())

In [ ]:
!python scripts/run_overlap_baseline.py \
  --config configs/overlap_baseline.yaml \
  --override kaggle_phase4.yaml \
  --validate-only

!python scripts/run_overlap_baseline.py \
  --config configs/overlap_baseline.yaml \
  --override kaggle_phase4.yaml \
  --max-samples 1

After inspecting the one-mixture output, remove `--max-samples 1` to run the development test split. Then continue with `notebooks/12_adaptive_e2e_surt.ipynb`.